In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

# -----------------------------
# Create Spark Session
# -----------------------------
spark = SparkSession.builder.appName("DataQualityChecks").getOrCreate()

# -----------------------------
# Read Dataset
# -----------------------------
df = spark.read.csv(
    "/mnt/outputs/school_performance_clean.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully")
print("Total Rows:", df.count())
df.printSchema()


# -----------------------------
# DQ-001 avg_score must not be NULL
# -----------------------------
def dq_001_avg_score_not_null(df):
    violations = df.filter(col("avg_score").isNull()).count()

    if violations > 0:
        raise ValueError("DQ-001 Failed: avg_score contains NULL values")

    return violations


# -----------------------------
# DQ-002 pass_rate must be between 0 and 1
# -----------------------------
def dq_002_pass_rate_range(df):
    violations = df.filter(
        (col("pass_rate") < 0) | (col("pass_rate") > 1)
    ).count()

    return violations


# -----------------------------
# DQ-003 exam_date format check
# -----------------------------
def dq_003_exam_date_format(df):
    violations = df.filter(
        to_date(col("exam_date"), "yyyy-MM-dd").isNull()
    ).count()

    return violations


# -----------------------------
# DQ-004 record_id must be unique
# -----------------------------
def dq_004_record_id_unique(df):

    total_records = df.count()
    unique_records = df.select("record_id").distinct().count()

    if total_records != unique_records:
        raise ValueError("DQ-004 Failed: Duplicate record_id found")

    return total_records - unique_records


# -----------------------------
# DQ-005 num_students must be positive
# -----------------------------
def dq_005_num_students_positive(df):

    violations = df.filter(col("num_students") <= 0).count()

    return violations


# -----------------------------
# DQ-006 school_id and teacher_id not NULL
# -----------------------------
def dq_006_ids_not_null(df):

    violations = df.filter(
        col("school_id").isNull()
        | col("teacher_id").isNull()
        | (col("school_id") == "")
        | (col("teacher_id") == "")
    ).count()

    return violations


# -----------------------------
# Run All Data Quality Checks
# -----------------------------
def run_all_checks(df):

    results = []

    results.append(("DQ-001", "avg_score NOT NULL", dq_001_avg_score_not_null(df)))
    results.append(("DQ-002", "pass_rate range 0–1", dq_002_pass_rate_range(df)))
    results.append(("DQ-003", "exam_date valid format", dq_003_exam_date_format(df)))
    results.append(("DQ-004", "record_id uniqueness", dq_004_record_id_unique(df)))
    results.append(("DQ-005", "num_students positive", dq_005_num_students_positive(df)))
    results.append(("DQ-006", "IDs not null/empty", dq_006_ids_not_null(df)))

    passed = 0

    print("\nData Quality Check Report\n")

    for rule, desc, violations in results:

        if violations == 0:
            print(f"{rule}  {desc:<25} PASS  ({violations} violations)")
            passed += 1
        else:
            print(f"{rule}  {desc:<25} FAIL  ({violations} violations)")

    print(f"\nOverall: {passed}/6 checks passed")

run_all_checks(df)

Dataset Loaded Successfully
Total Rows: 300
root
 |-- record_id: integer (nullable = true)
 |-- school_id: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- grade_level: string (nullable = true)
 |-- teacher_id: string (nullable = true)
 |-- teacher_name: string (nullable = true)
 |-- num_students: integer (nullable = true)
 |-- avg_score: double (nullable = true)
 |-- pass_rate: double (nullable = true)
 |-- exam_date: date (nullable = true)


Data Quality Check Report

DQ-001  avg_score NOT NULL        PASS  (0 violations)
DQ-002  pass_rate range 0–1       PASS  (0 violations)
DQ-003  exam_date valid format    PASS  (0 violations)
DQ-004  record_id uniqueness      PASS  (0 violations)
DQ-005  num_students positive     PASS  (0 violations)
DQ-006  IDs not null/empty        PASS  (0 violations)

Overall: 6/6 checks passed
